In [6]:
import xarray as xr

In [7]:
nlcd_ds = (
    xr.open_dataset("/home/mitchhung/data/nlcd/nlcd.zarr", chunks="auto")
    .assign(
        water=lambda ds: ds.landcover.isin([11, 12]).astype("uint8"),
        developed=lambda ds: ds.landcover.isin([21, 22, 23, 24]).astype("uint8"),
        barren=lambda ds: (ds.landcover == 31).astype("uint8"),
        deciduous_forest=lambda ds: (ds.landcover == 41).astype("uint8"),
        evergreen_forest=lambda ds: (ds.landcover == 42).astype("uint8"),
        mixed_forest=lambda ds: (ds.landcover == 43).astype("uint8"),
        shrub=lambda ds: (ds.landcover == 52).astype("uint8"),
        grass=lambda ds: (ds.landcover == 71).astype("uint8"),
        crops=lambda ds: ds.landcover.isin([81, 82]).astype("uint8"),
        wetlands=lambda ds: ds.landcover.isin([90, 95]).astype("uint8"),
    )
    .rio.write_crs("EPSG:4326")
    .rename({"lat": "y", "lon": "x"})
    # .sel(year=2017)
)
nlcd_ds

<xarray.Dataset> Size: 12TB
Dimensions:           (year: 39, y: 105000, x: 160000)
Coordinates:
  * y                 (y) float64 840kB 24.51 24.51 24.51 ... 52.92 52.92 52.92
  * x                 (x) float64 1MB -123.3 -123.3 -123.3 ... -69.07 -69.07
  * year              (year) int64 312B 1985 1986 1987 1988 ... 2021 2022 2023
    spatial_ref       int64 8B 0
Data variables:
    landcover         (year, y, x) float64 5TB dask.array<chunksize=(2, 2000, 2000), meta=np.ndarray>
    water             (year, y, x) uint8 655GB dask.array<chunksize=(2, 2000, 2000), meta=np.ndarray>
    developed         (year, y, x) uint8 655GB dask.array<chunksize=(2, 2000, 2000), meta=np.ndarray>
    barren            (year, y, x) uint8 655GB dask.array<chunksize=(2, 2000, 2000), meta=np.ndarray>
    deciduous_forest  (year, y, x) uint8 655GB dask.array<chunksize=(2, 2000, 2000), meta=np.ndarray>
    evergreen_forest  (year, y, x) uint8 655GB dask.array<chunksize=(2, 2000, 2000), meta=np.ndarray>
    mixed_forest      (year, y, x) uint8 655GB dask.array<chunksize=(2, 2000, 2000), meta=np.ndarray>
    shrub             (year, y, x) uint8 655GB dask.array<chunksize=(2, 2000, 2000), meta=np.ndarray>
    grass             (year, y, x) uint8 655GB dask.array<chunksize=(2, 2000, 2000), meta=np.ndarray>
    crops             (year, y, x) uint8 655GB dask.array<chunksize=(2, 2000, 2000), meta=np.ndarray>
    wetlands          (year, y, x) uint8 655GB dask.array<chunksize=(2, 2000, 2000), meta=np.ndarray>

In [9]:
import xarray as xr
from dask.diagnostics import ProgressBar

# Approximate block size for 1 km (since NLCD = 30 m resolution)
block = 33

masks = [
    "water",
    "developed",
    "barren",
    "deciduous_forest",
    "evergreen_forest",
    "mixed_forest",
    "shrub",
    "grass",
    "crops",
    "wetlands",
]

counts = []

for var in masks:
    print(f"Aggregating {var}")
    with ProgressBar():
        out = (
            nlcd_ds[var]
            .coarsen(y=block, x=block, boundary="trim")  # 33x33 pixel bins
            .sum()
            .astype("int16")
            .to_dataset(name=var)
        ) 
        # / (block * block)  # Convert to fraction
        counts.append(out)

# Merge into a single dataset with counts per 1 km cell
nlcd_counts = xr.merge(counts)

nlcd_counts

Aggregating water
Aggregating developed
Aggregating barren
Aggregating deciduous_forest
Aggregating evergreen_forest
Aggregating mixed_forest
Aggregating shrub
Aggregating grass
Aggregating crops
Aggregating wetlands


<xarray.Dataset> Size: 12GB
Dimensions:           (y: 3181, x: 4848, year: 39)
Coordinates:
  * y                 (y) float64 25kB 24.52 24.53 24.53 ... 52.89 52.9 52.91
  * x                 (x) float64 39kB -123.3 -123.2 -123.2 ... -69.09 -69.08
  * year              (year) int64 312B 1985 1986 1987 1988 ... 2021 2022 2023
    spatial_ref       int64 8B 0
Data variables:
    water             (year, y, x) int16 1GB dask.array<chunksize=(2, 60, 60), meta=np.ndarray>
    developed         (year, y, x) int16 1GB dask.array<chunksize=(2, 60, 60), meta=np.ndarray>
    barren            (year, y, x) int16 1GB dask.array<chunksize=(2, 60, 60), meta=np.ndarray>
    deciduous_forest  (year, y, x) int16 1GB dask.array<chunksize=(2, 60, 60), meta=np.ndarray>
    evergreen_forest  (year, y, x) int16 1GB dask.array<chunksize=(2, 60, 60), meta=np.ndarray>
    mixed_forest      (year, y, x) int16 1GB dask.array<chunksize=(2, 60, 60), meta=np.ndarray>
    shrub             (year, y, x) int16 1GB dask.array<chunksize=(2, 60, 60), meta=np.ndarray>
    grass             (year, y, x) int16 1GB dask.array<chunksize=(2, 60, 60), meta=np.ndarray>
    crops             (year, y, x) int16 1GB dask.array<chunksize=(2, 60, 60), meta=np.ndarray>
    wetlands          (year, y, x) int16 1GB dask.array<chunksize=(2, 60, 60), meta=np.ndarray>

In [10]:
nlcd_counts = (
    nlcd_counts
    .rename({"y": "lat", "x": "lon"})
)

for var in masks:
    nlcd_counts[var] = nlcd_counts[var] / (block * block)

nlcd_counts

<xarray.Dataset> Size: 48GB
Dimensions:           (lat: 3181, lon: 4848, year: 39)
Coordinates:
  * lat               (lat) float64 25kB 24.52 24.53 24.53 ... 52.89 52.9 52.91
  * lon               (lon) float64 39kB -123.3 -123.2 -123.2 ... -69.09 -69.08
  * year              (year) int64 312B 1985 1986 1987 1988 ... 2021 2022 2023
    spatial_ref       int64 8B 0
Data variables:
    water             (year, lat, lon) float64 5GB dask.array<chunksize=(2, 60, 60), meta=np.ndarray>
    developed         (year, lat, lon) float64 5GB dask.array<chunksize=(2, 60, 60), meta=np.ndarray>
    barren            (year, lat, lon) float64 5GB dask.array<chunksize=(2, 60, 60), meta=np.ndarray>
    deciduous_forest  (year, lat, lon) float64 5GB dask.array<chunksize=(2, 60, 60), meta=np.ndarray>
    evergreen_forest  (year, lat, lon) float64 5GB dask.array<chunksize=(2, 60, 60), meta=np.ndarray>
    mixed_forest      (year, lat, lon) float64 5GB dask.array<chunksize=(2, 60, 60), meta=np.ndarray>
    shrub             (year, lat, lon) float64 5GB dask.array<chunksize=(2, 60, 60), meta=np.ndarray>
    grass             (year, lat, lon) float64 5GB dask.array<chunksize=(2, 60, 60), meta=np.ndarray>
    crops             (year, lat, lon) float64 5GB dask.array<chunksize=(2, 60, 60), meta=np.ndarray>
    wetlands          (year, lat, lon) float64 5GB dask.array<chunksize=(2, 60, 60), meta=np.ndarray>

In [16]:
from dask.diagnostics import ProgressBar

out_path = "/home/mitchhung/data/nlcd/nlcd_1km_counts.zarr"

for year in range(1988, 2024):
    print(f"Year {year}")
    # Select just this year (already shape (lat, lon), with a 'year' coord of length 1)
    year_ds = nlcd_counts.sel(year=slice(year, year))

    if year > 1985:
        write_args = {
            "mode": "a",
            "append_dim": "year",
        }
    else:
        write_args = {"mode": "w"}

    with ProgressBar():
        year_ds.chunk({"year": 1, "lat": 1000, "lon": 1000}).to_zarr(
            out_path,
            **write_args
        )


Year 1988
[########################################] | 100% Completed | 325.95 s
Year 1989
[########################################] | 100% Completed | 284.67 s
Year 1990
[########################################] | 100% Completed | 284.54 s
Year 1991
[########################################] | 100% Completed | 284.01 s
Year 1992
[########################################] | 100% Completed | 284.76 s
Year 1993
[########################################] | 100% Completed | 288.91 s
Year 1994
[########################################] | 100% Completed | 284.87 s
Year 1995
[########################################] | 100% Completed | 283.62 s
Year 1996
[########################################] | 100% Completed | 285.28 s
Year 1997
[########################################] | 100% Completed | 288.14 s
Year 1998
[########################################] | 100% Completed | 284.53 s
Year 1999
[########################################] | 100% Completed | 287.11 s
Year 2000
[#################

In [15]:
ds = xr.open_dataset("/home/mitchhung/data/nlcd/nlcd_1km_counts.zarr")
ds

<xarray.Dataset> Size: 4GB
Dimensions:           (year: 3, lat: 3181, lon: 4848)
Coordinates:
  * lat               (lat) float64 25kB 24.52 24.53 24.53 ... 52.89 52.9 52.91
  * lon               (lon) float64 39kB -123.3 -123.2 -123.2 ... -69.09 -69.08
    spatial_ref       int64 8B ...
  * year              (year) int64 24B 1985 1986 1987
Data variables:
    barren            (year, lat, lon) float64 370MB ...
    crops             (year, lat, lon) float64 370MB ...
    deciduous_forest  (year, lat, lon) float64 370MB ...
    developed         (year, lat, lon) float64 370MB ...
    evergreen_forest  (year, lat, lon) float64 370MB ...
    grass             (year, lat, lon) float64 370MB ...
    mixed_forest      (year, lat, lon) float64 370MB ...
    shrub             (year, lat, lon) float64 370MB ...
    water             (year, lat, lon) float64 370MB ...
    wetlands          (year, lat, lon) float64 370MB ...